# MACD-BB Screener — NonKYC Public REST

This notebook screens **NONKYC** spot markets for **MACD + Bollinger Bands** using only public market-data
endpoints. It mirrors the public **PMM Dynamic** screener workflow and output shape, then layers on MACD-BB
signal diagnostics plus starter `macd_bb_v1` controller YAMLs sourced from `controller_yml_data_dictionary.md`.

Treat it as a **research / paper-trade triage tool**, not proof of live readiness. The exports include a ranked
shortlist, selected `BASE-QUOTE` pairs, candle-ingestor manifest, exchange-rules patch, controller template
bundle, and a markdown report with blockers and next checks.


In [1]:
import os
import sys
import subprocess
import logging
import math
import json
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 160)

SEARCH_ROOTS = [Path("/quants-lab"), Path("/mnt/data/quants-lab"), Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        p.resolve()
        for p in SEARCH_ROOTS
        if p.exists()
        and (p / "controller_yml_data_dictionary.md").exists()
        and (p / "research_notebooks" / "market_lab" / "pmm_dynamic").exists()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repo root. Open this notebook from inside the quants-lab repo or place it where "
        "controller_yml_data_dictionary.md and research_notebooks/market_lab/pmm_dynamic are reachable."
    )

PMM_DIR = REPO_ROOT / "research_notebooks" / "market_lab" / "pmm_dynamic"
if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))

EDITABLE_INSTALL_OK = False
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    EDITABLE_INSTALL_OK = True
except Exception as exc:
    print(f"Editable install skipped: {exc}")

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"Repo root: {REPO_ROOT}")
print(f"PMM project root: {PMM_DIR}")
print(f"Editable install ok: {EDITABLE_INSTALL_OK}")

from pmm_lab.screener import (
    MEXCPublicScreener,
    NonKYCPublicScreener,
    default_mexc_config,
    default_nonkyc_config,
    export_screening_artifacts,
)
from pmm_lab.screener.common import (
    ScreenerConfig,
    ScreeningRun,
    compute_coarse_scores,
    select_shortlist,
    compute_orderbook_metrics,
    compute_candle_metrics,
    compute_trade_metrics,
    percentile_rank,
    band_score,
    apply_screening_logic as apply_microstructure_screening,
    safe_float,
    now_utc_iso,
    hb_to_slash_pair,
    json_ready_records,
)
from pmm_lab.screener.mexc_public import MEXC_BASE_URL
from pmm_lab.screener.nonkyc_public import NONKYC_BASE_URL

DICT_PATH = REPO_ROOT / "controller_yml_data_dictionary.md"

# ============================================================
# MACD-BB notebook helpers
# ============================================================
from dataclasses import dataclass, field
from typing import Any, Mapping, Optional
import ast
import math
import re
import json
import textwrap

import numpy as np
import pandas as pd

@dataclass
class StrategyScreenConfig:
    interval: str
    interval_seconds: int
    bb_length: int
    bb_std: float
    bb_long_threshold: float
    bb_short_threshold: float
    macd_fast: int
    macd_slow: int
    macd_signal: int
    cooldown_time: int
    take_profit: float
    stop_loss: float
    time_limit_sec: int
    side_mode: str = "long_only"   # "long_only", "short_only", "both"
    min_signal_events: int = 3
    min_events_per_day: float = 0.0
    min_hit_rate: float = 0.0
    min_mean_net_edge_bps: float = 0.0
    min_profit_factor_proxy: float = 0.95
    max_ambiguous_rate: float = 0.50
    strategy_score_weight: float = 0.40


@dataclass
class ControllerTemplateConfig:
    connector_name: str
    candles_connector: Optional[str]
    interval: str
    total_amount_quote: float
    max_executors_per_side: int
    cooldown_time: int
    leverage: int
    position_mode: str
    stop_loss: Optional[float]
    take_profit: Optional[float]
    time_limit: Optional[int]
    take_profit_order_type: int
    trailing_stop: Optional[dict[str, float]]
    bb_length: int
    bb_std: float
    bb_long_threshold: float
    bb_short_threshold: float
    macd_fast: int
    macd_slow: int
    macd_signal: int
    controller_name: str = "macd_bb_v1"
    controller_type: str = "directional_trading"
    manual_kill_switch: bool = False
    initial_positions: list[dict[str, Any]] = field(default_factory=list)


@dataclass(frozen=True)
class MACDBBArtifactPaths:
    root_dir: str
    universe_csv: str
    shortlist_csv: str
    final_csv: str
    selected_csv: str
    selected_pairs_txt: str
    selected_pairs_json: str
    symbol_metadata_json: str
    candle_ingestor_manifest_yaml: str
    exchange_rules_patch_yaml: str
    controller_templates_yaml: str
    controller_summary_csv: str
    screening_report_md: str
    run_metadata_json: str


def read_controller_dictionary_md(dict_path: str | Path, controller_name: str = "macd_bb_v1") -> pd.DataFrame:
    path = Path(dict_path)
    text = path.read_text(encoding="utf-8")
    pattern = rf"^##\s+`{re.escape(controller_name)}`\s*$([\s\S]*?)(?=^##\s+`|\Z)"
    match = re.search(pattern, text, flags=re.MULTILINE)
    if not match:
        raise ValueError(f"Could not locate controller section {controller_name!r} in {path}")
    section = match.group(1)
    table_lines = [line.strip() for line in section.splitlines() if line.strip().startswith("|")]
    if not table_lines:
        raise ValueError(f"No markdown table found for {controller_name!r} in {path}")
    header = [col.strip() for col in table_lines[0].strip("|").split("|")]
    rows: list[list[str]] = []
    for line in table_lines[1:]:
        if re.match(r"^\|\s*---", line):
            continue
        rows.append([col.strip() for col in line.strip("|").split("|")])
    frame = pd.DataFrame(rows, columns=header)
    frame.columns = [c.strip().lower().replace(" ", "_") for c in frame.columns]
    for col in frame.columns:
        frame[col] = frame[col].map(lambda x: str(x).strip().strip("`") if x is not None else x)
    return frame


def _safe_eval_default(value: Any) -> Any:
    if value is None:
        return None
    s = str(value).strip()
    if not s or s in {"—", "required"}:
        return None
    low = s.lower()
    if low == "null":
        return None
    if low == "false":
        return False
    if low == "true":
        return True
    if s == "[]":
        return []
    if re.fullmatch(r"-?\d+", s):
        try:
            return int(s)
        except Exception:
            pass
    if re.fullmatch(r"-?\d*\.\d+", s):
        try:
            return float(s)
        except Exception:
            pass
    if re.fullmatch(r"[\d\.\s\+\-\*\/\(\)]+", s):
        try:
            return eval(s, {"__builtins__": {}}, {})
        except Exception:
            pass
    if s.startswith("[") or s.startswith("{") or s.startswith("("):
        try:
            return ast.literal_eval(s)
        except Exception:
            pass
    return s


def controller_defaults_from_dictionary(controller_dict_df: pd.DataFrame) -> dict[str, Any]:
    top_level = controller_dict_df[
        ~controller_dict_df["field_path"].astype(str).str.contains(r"\.")
        & ~controller_dict_df["field_path"].astype(str).str.contains(r"\[\]")
    ].copy()
    defaults = {
        str(row["field_path"]): _safe_eval_default(row.get("default"))
        for _, row in top_level.iterrows()
    }
    return defaults


def normalize_indicator_frame(candles: pd.DataFrame) -> pd.DataFrame:
    required = ["timestamp", "open", "high", "low", "close", "volume"]
    if candles is None or candles.empty:
        return pd.DataFrame(columns=required + ["datetime"])
    df = candles.copy()
    for col in required:
        if col not in df.columns:
            df[col] = np.nan
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df.loc[df["timestamp"] > 1e12, "timestamp"] = df.loc[df["timestamp"] > 1e12, "timestamp"] / 1000.0
    for col in ("open", "high", "low", "close", "volume"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=required).sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True)
    if df.empty:
        return pd.DataFrame(columns=required + ["datetime"])
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    return df


def _find_first_column(df: pd.DataFrame, prefix: str) -> str:
    matches = [c for c in df.columns if str(c).startswith(prefix)]
    if not matches:
        raise KeyError(f"Could not locate indicator column with prefix {prefix!r}")
    return matches[0]


def add_macd_bb_signals(candles: pd.DataFrame, strategy_cfg: StrategyScreenConfig) -> pd.DataFrame:
    df = normalize_indicator_frame(candles)
    if df.empty:
        return df

    close = pd.to_numeric(df["close"], errors="coerce")

    # Bollinger Band %B (controller logic uses BBP from pandas_ta)
    bb_length = int(strategy_cfg.bb_length)
    bb_std = float(strategy_cfg.bb_std)
    mid = close.rolling(window=bb_length, min_periods=bb_length).mean()
    dev = close.rolling(window=bb_length, min_periods=bb_length).std(ddof=0)
    upper = mid + (bb_std * dev)
    lower = mid - (bb_std * dev)
    width = (upper - lower).replace(0, np.nan)
    df["bbp_value"] = (close - lower) / width

    # MACD and histogram
    fast = int(strategy_cfg.macd_fast)
    slow = int(strategy_cfg.macd_slow)
    signal = int(strategy_cfg.macd_signal)
    ema_fast = close.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = close.ewm(span=slow, adjust=False, min_periods=slow).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False, min_periods=signal).mean()
    macdh_line = macd_line - signal_line

    df["macd_value"] = macd_line
    df["macdh_value"] = macdh_line

    long_condition = (
        (df["bbp_value"] < float(strategy_cfg.bb_long_threshold))
        & (df["macdh_value"] > 0)
        & (df["macd_value"] < 0)
    )
    short_condition = (
        (df["bbp_value"] > float(strategy_cfg.bb_short_threshold))
        & (df["macdh_value"] < 0)
        & (df["macd_value"] > 0)
    )
    df["long_signal"] = long_condition.fillna(False).astype(int)
    df["short_signal"] = short_condition.fillna(False).astype(int)
    df["signal"] = 0
    df.loc[long_condition.fillna(False), "signal"] = 1
    df.loc[short_condition.fillna(False), "signal"] = -1
    return df

def evaluate_barrier_path(
    signal_df: pd.DataFrame,
    entry_idx: int,
    side: int,
    take_profit: float,
    stop_loss: float,
    time_limit_bars: int,
) -> dict[str, Any]:
    if entry_idx >= len(signal_df) - 1:
        return {
            "outcome": "insufficient_forward_bars",
            "exit_idx": entry_idx,
            "exit_timestamp": safe_float(signal_df.iloc[entry_idx]["timestamp"]),
            "exit_price": safe_float(signal_df.iloc[entry_idx]["close"]),
            "barrier_return": 0.0,
            "forward_return": 0.0,
            "best_excursion": 0.0,
            "worst_excursion": 0.0,
        }

    entry_row = signal_df.iloc[entry_idx]
    entry_price = safe_float(entry_row["close"])
    if math.isnan(entry_price) or entry_price <= 0:
        return {
            "outcome": "invalid_entry_price",
            "exit_idx": entry_idx,
            "exit_timestamp": safe_float(entry_row["timestamp"]),
            "exit_price": math.nan,
            "barrier_return": math.nan,
            "forward_return": math.nan,
            "best_excursion": math.nan,
            "worst_excursion": math.nan,
        }

    end_idx = min(entry_idx + max(int(time_limit_bars), 1), len(signal_df) - 1)
    future = signal_df.iloc[entry_idx + 1 : end_idx + 1].copy()
    if future.empty:
        return {
            "outcome": "insufficient_forward_bars",
            "exit_idx": entry_idx,
            "exit_timestamp": safe_float(entry_row["timestamp"]),
            "exit_price": entry_price,
            "barrier_return": 0.0,
            "forward_return": 0.0,
            "best_excursion": 0.0,
            "worst_excursion": 0.0,
        }

    if side > 0:
        take_level = entry_price * (1.0 + float(take_profit))
        stop_level = entry_price * (1.0 - float(stop_loss))
        favorable_path = (future["high"] / entry_price) - 1.0
        adverse_path = (future["low"] / entry_price) - 1.0
    else:
        take_level = entry_price * (1.0 - float(take_profit))
        stop_level = entry_price * (1.0 + float(stop_loss))
        favorable_path = 1.0 - (future["low"] / entry_price)
        adverse_path = 1.0 - (future["high"] / entry_price)

    best_excursion = float(pd.to_numeric(favorable_path, errors="coerce").max(skipna=True))
    worst_excursion = float(pd.to_numeric(adverse_path, errors="coerce").min(skipna=True))

    outcome = "time_limit"
    exit_idx = end_idx
    exit_timestamp = safe_float(signal_df.iloc[end_idx]["timestamp"])
    exit_price = safe_float(signal_df.iloc[end_idx]["close"])
    barrier_return = float(side * ((exit_price / entry_price) - 1.0))
    forward_price = exit_price
    forward_return = float(side * ((forward_price / entry_price) - 1.0))

    for j in range(entry_idx + 1, end_idx + 1):
        row = signal_df.iloc[j]
        high = safe_float(row["high"])
        low = safe_float(row["low"])
        ts = safe_float(row["timestamp"])
        if side > 0:
            tp_hit = high >= take_level
            sl_hit = low <= stop_level
        else:
            tp_hit = low <= take_level
            sl_hit = high >= stop_level

        if tp_hit and sl_hit:
            outcome = "ambiguous"
            exit_idx = j
            exit_timestamp = ts
            exit_price = stop_level
            barrier_return = -float(stop_loss)
            break
        if tp_hit:
            outcome = "take_profit"
            exit_idx = j
            exit_timestamp = ts
            exit_price = take_level
            barrier_return = float(take_profit)
            break
        if sl_hit:
            outcome = "stop_loss"
            exit_idx = j
            exit_timestamp = ts
            exit_price = stop_level
            barrier_return = -float(stop_loss)
            break

    return {
        "outcome": outcome,
        "exit_idx": int(exit_idx),
        "exit_timestamp": exit_timestamp,
        "exit_price": exit_price,
        "barrier_return": float(barrier_return),
        "forward_return": float(forward_return),
        "best_excursion": best_excursion,
        "worst_excursion": worst_excursion,
    }


def extract_signal_events(signal_df: pd.DataFrame, strategy_cfg: StrategyScreenConfig) -> pd.DataFrame:
    if signal_df is None or signal_df.empty or "signal" not in signal_df.columns:
        return pd.DataFrame()
    cooldown_bars = max(int(math.ceil(float(strategy_cfg.cooldown_time) / max(strategy_cfg.interval_seconds, 1))), 1)
    records: list[dict[str, Any]] = []
    last_seen = {1: -10**9, -1: -10**9}
    for idx, row in signal_df.iterrows():
        side = int(row.get("signal", 0) or 0)
        if side not in (1, -1):
            continue
        if idx - last_seen[side] < cooldown_bars:
            continue
        last_seen[side] = int(idx)
        records.append(
            {
                "entry_idx": int(idx),
                "entry_timestamp": safe_float(row["timestamp"]),
                "entry_price": safe_float(row["close"]),
                "side": side,
                "signal_label": "long" if side > 0 else "short",
                "bbp_value": safe_float(row.get("bbp_value")),
                "macd_value": safe_float(row.get("macd_value")),
                "macdh_value": safe_float(row.get("macdh_value")),
            }
        )
    return pd.DataFrame(records)


def _profit_factor_proxy(values: pd.Series) -> float:
    s = pd.to_numeric(values, errors="coerce").dropna()
    if s.empty:
        return math.nan
    profits = float(s[s > 0].sum())
    losses = float(-s[s < 0].sum())
    if losses <= 0:
        return math.inf if profits > 0 else math.nan
    return profits / losses


def summarize_event_subset(
    events_df: pd.DataFrame,
    sample_days: float,
    spread_bps: float,
    prefix: str,
) -> dict[str, Any]:
    base = {
        f"{prefix}_signal_count": 0,
        f"{prefix}_events_per_day": 0.0,
        f"{prefix}_hit_rate": math.nan,
        f"{prefix}_take_profit_rate": math.nan,
        f"{prefix}_stop_loss_rate": math.nan,
        f"{prefix}_time_limit_rate": math.nan,
        f"{prefix}_ambiguous_rate": math.nan,
        f"{prefix}_mean_barrier_return_bps": math.nan,
        f"{prefix}_mean_barrier_return_net_bps": math.nan,
        f"{prefix}_mean_forward_return_bps": math.nan,
        f"{prefix}_profit_factor_proxy": math.nan,
        f"{prefix}_edge_to_spread_ratio": math.nan,
        f"{prefix}_best_excursion_bps": math.nan,
        f"{prefix}_worst_excursion_bps": math.nan,
    }
    if events_df is None or events_df.empty:
        return base

    barrier_ret = pd.to_numeric(events_df["barrier_return"], errors="coerce")
    forward_ret = pd.to_numeric(events_df["forward_return"], errors="coerce")
    best_exc = pd.to_numeric(events_df["best_excursion"], errors="coerce")
    worst_exc = pd.to_numeric(events_df["worst_excursion"], errors="coerce")
    outcomes = events_df["outcome"].astype(str)

    spread_bps_value = safe_float(spread_bps, default=0.0)
    mean_barrier_bps = float(barrier_ret.mean(skipna=True) * 10_000.0)
    mean_net_bps = mean_barrier_bps - (0.0 if math.isnan(spread_bps_value) else float(spread_bps_value))

    base.update(
        {
            f"{prefix}_signal_count": int(len(events_df)),
            f"{prefix}_events_per_day": float(len(events_df) / max(sample_days, 1e-9)),
            f"{prefix}_hit_rate": float((barrier_ret > 0).mean()),
            f"{prefix}_take_profit_rate": float((outcomes == "take_profit").mean()),
            f"{prefix}_stop_loss_rate": float((outcomes == "stop_loss").mean()),
            f"{prefix}_time_limit_rate": float((outcomes == "time_limit").mean()),
            f"{prefix}_ambiguous_rate": float((outcomes == "ambiguous").mean()),
            f"{prefix}_mean_barrier_return_bps": mean_barrier_bps,
            f"{prefix}_mean_barrier_return_net_bps": mean_net_bps,
            f"{prefix}_mean_forward_return_bps": float(forward_ret.mean(skipna=True) * 10_000.0),
            f"{prefix}_profit_factor_proxy": float(_profit_factor_proxy(barrier_ret)),
            f"{prefix}_edge_to_spread_ratio": (
                float(mean_barrier_bps / spread_bps_value)
                if not math.isnan(spread_bps_value) and spread_bps_value > 0
                else math.nan
            ),
            f"{prefix}_best_excursion_bps": float(best_exc.mean(skipna=True) * 10_000.0),
            f"{prefix}_worst_excursion_bps": float(worst_exc.mean(skipna=True) * 10_000.0),
        }
    )
    return base


def compute_macd_bb_strategy_metrics(
    candles: pd.DataFrame,
    strategy_cfg: StrategyScreenConfig,
    spread_bps: float = math.nan,
) -> dict[str, Any]:
    template = {
        "strategy_side_mode": strategy_cfg.side_mode,
        "strategy_sample_days": math.nan,
        "strategy_time_limit_bars": math.nan,
        "strategy_current_signal": 0,
        "strategy_current_bbp": math.nan,
        "strategy_current_macd": math.nan,
        "strategy_current_macdh": math.nan,
        "strategy_last_signal_ts": math.nan,
        "strategy_last_signal_side": "",
        "strategy_bars_since_last_signal": math.nan,
        "strategy_recommended_side": "",
        "strategy_signal_count": 0,
        "strategy_events_per_day": 0.0,
        "strategy_hit_rate": math.nan,
        "strategy_take_profit_rate": math.nan,
        "strategy_stop_loss_rate": math.nan,
        "strategy_time_limit_rate": math.nan,
        "strategy_ambiguous_rate": math.nan,
        "strategy_mean_barrier_return_bps": math.nan,
        "strategy_mean_barrier_return_net_bps": math.nan,
        "strategy_mean_forward_return_bps": math.nan,
        "strategy_profit_factor_proxy": math.nan,
        "strategy_edge_to_spread_ratio": math.nan,
        "strategy_best_excursion_bps": math.nan,
        "strategy_worst_excursion_bps": math.nan,
    }
    template.update(summarize_event_subset(pd.DataFrame(), 1.0, spread_bps, "long"))
    template.update(summarize_event_subset(pd.DataFrame(), 1.0, spread_bps, "short"))

    signal_df = add_macd_bb_signals(candles, strategy_cfg)
    if signal_df.empty:
        return template

    sample_days = max(
        float((signal_df["timestamp"].iloc[-1] - signal_df["timestamp"].iloc[0]) / 86400.0),
        float(len(signal_df) * strategy_cfg.interval_seconds / 86400.0),
    )
    time_limit_bars = max(int(math.ceil(float(strategy_cfg.time_limit_sec) / max(strategy_cfg.interval_seconds, 1))), 1)

    template.update(
        {
            "strategy_sample_days": sample_days,
            "strategy_time_limit_bars": time_limit_bars,
            "strategy_current_signal": int(signal_df["signal"].iloc[-1]),
            "strategy_current_bbp": safe_float(signal_df["bbp_value"].iloc[-1]),
            "strategy_current_macd": safe_float(signal_df["macd_value"].iloc[-1]),
            "strategy_current_macdh": safe_float(signal_df["macdh_value"].iloc[-1]),
        }
    )

    events = extract_signal_events(signal_df, strategy_cfg)
    if events.empty:
        return template

    evaluated_rows: list[dict[str, Any]] = []
    for _, event in events.iterrows():
        barrier = evaluate_barrier_path(
            signal_df,
            entry_idx=int(event["entry_idx"]),
            side=int(event["side"]),
            take_profit=float(strategy_cfg.take_profit),
            stop_loss=float(strategy_cfg.stop_loss),
            time_limit_bars=time_limit_bars,
        )
        evaluated_rows.append({**event.to_dict(), **barrier})

    events_df = pd.DataFrame(evaluated_rows)
    if events_df.empty:
        return template

    last_signal = events_df.iloc[-1]
    template["strategy_last_signal_ts"] = safe_float(last_signal.get("entry_timestamp"))
    template["strategy_last_signal_side"] = str(last_signal.get("signal_label", ""))
    template["strategy_bars_since_last_signal"] = int(len(signal_df) - 1 - int(last_signal.get("entry_idx", len(signal_df) - 1)))

    long_df = events_df[events_df["side"] > 0].copy()
    short_df = events_df[events_df["side"] < 0].copy()

    template.update(summarize_event_subset(long_df, sample_days, spread_bps, "long"))
    template.update(summarize_event_subset(short_df, sample_days, spread_bps, "short"))

    long_net = safe_float(template["long_mean_barrier_return_net_bps"])
    short_net = safe_float(template["short_mean_barrier_return_net_bps"])
    if math.isnan(long_net) and math.isnan(short_net):
        recommended = ""
    elif math.isnan(short_net) or long_net >= short_net:
        recommended = "long"
    else:
        recommended = "short"
    template["strategy_recommended_side"] = recommended

    if strategy_cfg.side_mode == "long_only":
        active_df = long_df
    elif strategy_cfg.side_mode == "short_only":
        active_df = short_df
    else:
        active_df = events_df

    template.update(summarize_event_subset(active_df, sample_days, spread_bps, "strategy"))
    return template


def _extract_book_sides(orderbook: Any) -> tuple[Any, Any]:
    if isinstance(orderbook, Mapping):
        return orderbook.get("bids"), orderbook.get("asks")
    return None, None


def enrich_pair_with_macd_bb(
    screener,
    row: Mapping[str, Any],
    strategy_cfg: StrategyScreenConfig,
) -> dict[str, Any]:
    exchange_symbol = str(row["exchange_symbol"])
    orderbook = screener._fetch_orderbook(exchange_symbol)
    candles = screener._fetch_candles(exchange_symbol)
    trades = screener._fetch_trades(exchange_symbol)

    bids, asks = _extract_book_sides(orderbook)
    orderbook_metrics = compute_orderbook_metrics(bids, asks)
    candle_metrics = compute_candle_metrics(candles, interval_seconds=int(strategy_cfg.interval_seconds))
    trade_metrics = compute_trade_metrics(trades)
    strategy_metrics = compute_macd_bb_strategy_metrics(
        candles,
        strategy_cfg=strategy_cfg,
        spread_bps=safe_float(orderbook_metrics.get("spread_bps")),
    )

    return {
        **orderbook_metrics,
        **candle_metrics,
        **trade_metrics,
        **strategy_metrics,
    }


def build_macd_bb_rejection_reasons(
    row: Mapping[str, Any],
    strategy_cfg: StrategyScreenConfig,
) -> list[str]:
    reasons: list[str] = []

    signal_count = safe_float(row.get("strategy_signal_count"))
    if math.isnan(signal_count):
        reasons.append("missing_strategy_signal_count")
    elif signal_count < strategy_cfg.min_signal_events:
        reasons.append(f"strategy_signal_count<{strategy_cfg.min_signal_events:g}")

    events_per_day = safe_float(row.get("strategy_events_per_day"))
    if strategy_cfg.min_events_per_day > 0:
        if math.isnan(events_per_day):
            reasons.append("missing_strategy_events_per_day")
        elif events_per_day < strategy_cfg.min_events_per_day:
            reasons.append(f"strategy_events_per_day<{strategy_cfg.min_events_per_day:g}")

    hit_rate = safe_float(row.get("strategy_hit_rate"))
    if strategy_cfg.min_hit_rate > 0:
        if math.isnan(hit_rate):
            reasons.append("missing_strategy_hit_rate")
        elif hit_rate < strategy_cfg.min_hit_rate:
            reasons.append(f"strategy_hit_rate<{strategy_cfg.min_hit_rate:g}")

    mean_net = safe_float(row.get("strategy_mean_barrier_return_net_bps"))
    if math.isnan(mean_net):
        reasons.append("missing_strategy_mean_net_edge_bps")
    elif mean_net < strategy_cfg.min_mean_net_edge_bps:
        reasons.append(f"strategy_mean_net_edge_bps<{strategy_cfg.min_mean_net_edge_bps:g}")

    pf = safe_float(row.get("strategy_profit_factor_proxy"))
    if math.isnan(pf):
        reasons.append("missing_strategy_profit_factor_proxy")
    elif pf < strategy_cfg.min_profit_factor_proxy:
        reasons.append(f"strategy_profit_factor_proxy<{strategy_cfg.min_profit_factor_proxy:g}")

    amb = safe_float(row.get("strategy_ambiguous_rate"))
    if not math.isnan(amb) and amb > strategy_cfg.max_ambiguous_rate:
        reasons.append(f"strategy_ambiguous_rate>{strategy_cfg.max_ambiguous_rate:g}")

    return reasons


def apply_macd_bb_screening_logic(
    final_df: pd.DataFrame,
    market_cfg: ScreenerConfig,
    strategy_cfg: StrategyScreenConfig,
) -> pd.DataFrame:
    if final_df is None or final_df.empty:
        return pd.DataFrame()

    required_micro_cols = {
        "is_active": False,
        "quote_asset": "",
        "quote_volume_24h": math.nan,
        "spread_bps": math.nan,
        "top_of_book_quote": math.nan,
        "sym_depth_quote_10bps": math.nan,
        "sym_depth_quote_50bps": math.nan,
        "sym_depth_quote_1xspread": math.nan,
        "recent_trade_count": math.nan,
        "trades_per_minute": math.nan,
        "last_trade_age_sec": math.nan,
        "n_candles": math.nan,
        "coverage_ratio": math.nan,
        "zero_volume_fraction": math.nan,
        "natr_bps_mean": math.nan,
        "efficiency_ratio": math.nan,
    }
    prepared = final_df.copy()
    for col, default_value in required_micro_cols.items():
        if col not in prepared.columns:
            prepared[col] = default_value

    out = apply_microstructure_screening(prepared, market_cfg).copy()

    out["microstructure_screen_score"] = pd.to_numeric(out["screen_score"], errors="coerce")
    out["microstructure_passed_filters"] = out["passed_filters"].fillna(False).astype(bool)
    out["microstructure_rejection_reasons"] = out["rejection_reasons"].apply(
        lambda v: list(v) if isinstance(v, (list, tuple)) else []
    )
    out["microstructure_rejection_reason"] = out["rejection_reason"].fillna("").astype(str)

    edge_series = pd.to_numeric(out["strategy_mean_barrier_return_net_bps"], errors="coerce")
    pf_series = pd.to_numeric(out["strategy_profit_factor_proxy"], errors="coerce").replace(np.inf, np.nan)
    count_series = pd.to_numeric(out["strategy_signal_count"], errors="coerce")
    epd_series = pd.to_numeric(out["strategy_events_per_day"], errors="coerce")
    hit_series = pd.to_numeric(out["strategy_hit_rate"], errors="coerce")
    amb_series = pd.to_numeric(out["strategy_ambiguous_rate"], errors="coerce")

    out["strategy_event_rank"] = percentile_rank(np.log1p(count_series.clip(lower=0)))
    out["strategy_activity_rank"] = percentile_rank(epd_series.clip(lower=0))
    out["strategy_hit_rank"] = percentile_rank(hit_series)
    out["strategy_edge_rank"] = percentile_rank(edge_series)
    out["strategy_pf_rank"] = percentile_rank(np.log1p(pf_series.clip(lower=0)))
    out["strategy_ambiguity_rank"] = percentile_rank(amb_series, reverse=True)

    out["strategy_rank"] = (
        0.20 * out["strategy_event_rank"]
        + 0.10 * out["strategy_activity_rank"]
        + 0.20 * out["strategy_hit_rank"]
        + 0.30 * out["strategy_edge_rank"]
        + 0.10 * out["strategy_pf_rank"]
        + 0.10 * out["strategy_ambiguity_rank"]
    )
    out["strategy_score"] = 100.0 * out["strategy_rank"]

    strategy_weight = float(strategy_cfg.strategy_score_weight)
    strategy_weight = min(max(strategy_weight, 0.0), 1.0)
    out["screen_score"] = 100.0 * (
        (1.0 - strategy_weight) * (out["microstructure_screen_score"] / 100.0)
        + strategy_weight * out["strategy_rank"]
    )

    merged_reasons: list[list[str]] = []
    merged_reason_text: list[str] = []
    for row in out.to_dict(orient="records"):
        reasons = list(row.get("microstructure_rejection_reasons") or [])
        reasons.extend(build_macd_bb_rejection_reasons(row, strategy_cfg))
        reasons = list(dict.fromkeys([r for r in reasons if r]))
        merged_reasons.append(reasons)
        merged_reason_text.append("; ".join(reasons))

    out["rejection_reasons"] = merged_reasons
    out["rejection_reason"] = merged_reason_text
    out["passed_filters"] = [len(r) == 0 for r in merged_reasons]

    return out.sort_values(
        ["passed_filters", "screen_score", "microstructure_screen_score", "quote_volume_24h"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


def screen_macd_bb_from_universe(
    screener,
    universe: pd.DataFrame,
    market_cfg: ScreenerConfig,
    strategy_cfg: StrategyScreenConfig,
) -> ScreeningRun:
    started = now_utc_iso()
    universe = universe.copy()
    if universe.empty:
        empty = pd.DataFrame()
        return ScreeningRun(
            connector=market_cfg.connector,
            config=market_cfg,
            universe=universe,
            shortlist=empty,
            final=empty,
            selected=empty,
            started_at=started,
            finished_at=now_utc_iso(),
            notes=["Universe build returned zero rows."],
            selection_mode="strict",
        )

    if "coarse_score" not in universe.columns:
        universe = compute_coarse_scores(universe)
    shortlist = select_shortlist(universe, market_cfg)

    enriched_rows: list[dict[str, Any]] = []
    for _, row in shortlist.iterrows():
        row_dict = row.to_dict()
        try:
            metrics = enrich_pair_with_macd_bb(screener, row_dict, strategy_cfg)
        except Exception as exc:
            metrics = {
                "screen_error": str(exc),
                "passed_filters": False,
                "rejection_reason": f"enrichment_error:{exc}",
                "rejection_reasons": [f"enrichment_error:{exc}"],
            }
            logging.exception("Failed to enrich %s %s", market_cfg.connector, row_dict.get("trading_pair"))
        enriched_rows.append({**row_dict, **metrics})
        if market_cfg.request_pause_sec > 0:
            import time
            time.sleep(market_cfg.request_pause_sec)

    final_df = pd.DataFrame(enriched_rows)
    if not final_df.empty:
        final_df = apply_macd_bb_screening_logic(final_df, market_cfg, strategy_cfg)
    selected_df = final_df[final_df["passed_filters"]].copy() if not final_df.empty else final_df.copy()

    selection_mode = "strict"
    if not selected_df.empty:
        selected_df = selected_df.sort_values(
            ["screen_score", "microstructure_screen_score", "quote_volume_24h"],
            ascending=[False, False, False],
        ).head(market_cfg.final_top_n)
    elif market_cfg.fallback_if_empty and not final_df.empty:
        selection_mode = "fallback_ranked"
        selected_df = final_df.sort_values(
            ["screen_score", "microstructure_screen_score", "quote_volume_24h"],
            ascending=[False, False, False],
        ).head(market_cfg.final_top_n).copy()

    notes = [
        "Research / paper-trade triage only. Passing the screener does not imply live readiness.",
        "Strategy metrics are approximate candle-path diagnostics, not full execution-aware backtests.",
        "Round-trip fees are not modeled; net-edge figures subtract observed spread only.",
    ]
    if strategy_cfg.side_mode == "long_only":
        notes.append("Default side_mode is long_only because these public screeners use spot market data.")
    if selection_mode == "fallback_ranked":
        notes.append(
            "WARNING: No pairs passed strict hard gates. Selected set is fallback_ranked. Review rejection_reason carefully."
        )

    return ScreeningRun(
        connector=market_cfg.connector,
        config=market_cfg,
        universe=universe,
        shortlist=shortlist,
        final=final_df,
        selected=selected_df,
        started_at=started,
        finished_at=now_utc_iso(),
        notes=notes,
        selection_mode=selection_mode,
    )


def build_controller_templates(
    run: ScreeningRun,
    controller_template_cfg: ControllerTemplateConfig,
    controller_dict_df: pd.DataFrame,
) -> list[dict[str, Any]]:
    defaults = controller_defaults_from_dictionary(controller_dict_df)
    top_level_fields = set(
        controller_dict_df.loc[
            ~controller_dict_df["field_path"].astype(str).str.contains(r"\.")
            & ~controller_dict_df["field_path"].astype(str).str.contains(r"\[\]"),
            "field_path",
        ].astype(str)
    )

    templates: list[dict[str, Any]] = []
    if run.selected is None or run.selected.empty:
        return templates

    for _, row in run.selected.iterrows():
        pair = str(row["trading_pair"])
        connector = str(controller_template_cfg.connector_name or run.connector)
        payload = {k: defaults.get(k) for k in top_level_fields if k in defaults}
        payload.update(
            {
                "id": f"macd_bb_{connector}_{pair.lower().replace('-', '_')}_{controller_template_cfg.interval}",
                "controller_name": controller_template_cfg.controller_name,
                "controller_type": controller_template_cfg.controller_type,
                "manual_kill_switch": bool(controller_template_cfg.manual_kill_switch),
                "initial_positions": list(controller_template_cfg.initial_positions),
                "connector_name": connector,
                "trading_pair": pair,
                "total_amount_quote": float(controller_template_cfg.total_amount_quote),
                "max_executors_per_side": int(controller_template_cfg.max_executors_per_side),
                "cooldown_time": int(controller_template_cfg.cooldown_time),
                "leverage": int(controller_template_cfg.leverage),
                "position_mode": str(controller_template_cfg.position_mode),
                "stop_loss": (
                    float(controller_template_cfg.stop_loss)
                    if controller_template_cfg.stop_loss is not None
                    else None
                ),
                "take_profit": (
                    float(controller_template_cfg.take_profit)
                    if controller_template_cfg.take_profit is not None
                    else None
                ),
                "time_limit": (
                    int(controller_template_cfg.time_limit)
                    if controller_template_cfg.time_limit is not None
                    else None
                ),
                "take_profit_order_type": int(controller_template_cfg.take_profit_order_type),
                "trailing_stop": controller_template_cfg.trailing_stop,
                "candles_connector": controller_template_cfg.candles_connector or connector,
                "candles_trading_pair": pair,
                "interval": controller_template_cfg.interval,
                "bb_length": int(controller_template_cfg.bb_length),
                "bb_std": float(controller_template_cfg.bb_std),
                "bb_long_threshold": float(controller_template_cfg.bb_long_threshold),
                "bb_short_threshold": float(controller_template_cfg.bb_short_threshold),
                "macd_fast": int(controller_template_cfg.macd_fast),
                "macd_slow": int(controller_template_cfg.macd_slow),
                "macd_signal": int(controller_template_cfg.macd_signal),
            }
        )
        payload = {k: v for k, v in payload.items() if k in top_level_fields}
        templates.append(payload)

    return templates


def build_controller_summary(run: ScreeningRun) -> pd.DataFrame:
    if run.selected is None or run.selected.empty:
        return pd.DataFrame()
    keep_cols = [
        "trading_pair",
        "screen_score",
        "microstructure_screen_score",
        "strategy_score",
        "strategy_signal_count",
        "strategy_hit_rate",
        "strategy_mean_barrier_return_net_bps",
        "strategy_profit_factor_proxy",
        "strategy_ambiguous_rate",
        "strategy_current_signal",
        "strategy_recommended_side",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
    ]
    cols = [c for c in keep_cols if c in run.selected.columns]
    return run.selected.loc[:, cols].copy()


def build_symbol_metadata(run: ScreeningRun) -> list[dict[str, Any]]:
    if run.selected is None or run.selected.empty:
        return []
    keep_cols = [
        "connector",
        "exchange_symbol",
        "trading_pair",
        "base_asset",
        "quote_asset",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "sym_depth_quote_50bps",
        "sym_depth_quote_1xspread",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "screen_score",
        "microstructure_screen_score",
        "strategy_score",
        "strategy_signal_count",
        "strategy_hit_rate",
        "strategy_mean_barrier_return_net_bps",
        "strategy_profit_factor_proxy",
        "strategy_ambiguous_rate",
        "strategy_current_signal",
        "strategy_recommended_side",
        "price_tick_estimate",
        "amount_step_estimate",
        "min_notional_quote_estimate",
        "min_order_size_base_estimate",
        "rule_estimate_source",
        "passed_filters",
        "rejection_reason",
    ]
    cols = [c for c in keep_cols if c in run.selected.columns]
    return json_ready_records(run.selected.loc[:, cols].copy())




def _frame_to_markdown(df: pd.DataFrame, index: bool = False) -> str:
    if df is None or df.empty:
        return "_No rows._"
    try:
        return df.to_markdown(index=index)
    except Exception:
        return "```\n" + df.to_csv(index=index) + "\n```"


def build_screening_report(
    run: ScreeningRun,
    controller_dict_df: pd.DataFrame,
    strategy_cfg: StrategyScreenConfig,
    controller_template_cfg: ControllerTemplateConfig,
) -> str:
    selected = run.selected.copy() if run.selected is not None else pd.DataFrame()
    final_df = run.final.copy() if run.final is not None else pd.DataFrame()

    selected_cols = [
        c for c in [
            "trading_pair",
            "screen_score",
            "strategy_score",
            "strategy_signal_count",
            "strategy_hit_rate",
            "strategy_mean_barrier_return_net_bps",
            "strategy_profit_factor_proxy",
            "spread_bps",
            "quote_volume_24h",
            "strategy_current_signal",
            "rejection_reason",
        ]
        if c in selected.columns
    ]
    selected_md = (
        _frame_to_markdown(selected.loc[:, selected_cols].head(25), index=False)
        if not selected.empty and selected_cols
        else "_No selected pairs._"
    )

    reject_md = "_No rejection summary available._"
    if not final_df.empty and "rejection_reason" in final_df.columns:
        reject_counts = (
            final_df.loc[~final_df["passed_filters"], "rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        if not reject_counts.empty:
            reject_md = _frame_to_markdown(reject_counts.head(20), index=True)

    dict_rows = controller_dict_df.loc[
        controller_dict_df["field_path"].isin(
            [
                "controller_name",
                "controller_type",
                "total_amount_quote",
                "connector_name",
                "trading_pair",
                "max_executors_per_side",
                "cooldown_time",
                "leverage",
                "position_mode",
                "stop_loss",
                "take_profit",
                "time_limit",
                "take_profit_order_type",
                "trailing_stop",
                "candles_connector",
                "candles_trading_pair",
                "interval",
                "bb_length",
                "bb_std",
                "bb_long_threshold",
                "bb_short_threshold",
                "macd_fast",
                "macd_slow",
                "macd_signal",
            ]
        ),
        ["field_path", "default", "what_it_does"],
    ].copy()
    dict_md = _frame_to_markdown(dict_rows, index=False) if not dict_rows.empty else "_Dictionary rows unavailable._"

    report = f"""
# MACD-BB public screener report

## Executive summary

- Connector: `{run.connector}`
- Screening mode: `{strategy_cfg.side_mode}`
- Universe rows: `{len(run.universe) if run.universe is not None else 0}`
- Shortlist rows: `{len(run.shortlist) if run.shortlist is not None else 0}`
- Enriched rows: `{len(run.final) if run.final is not None else 0}`
- Selected rows: `{len(run.selected) if run.selected is not None else 0}`
- Selection mode: `{run.selection_mode}`
- Fitness: **research / paper-trade triage only**
- Main live blockers:
  1. public REST candles + order book snapshots do not model queue position, latency, partial fills, or fees
  2. barrier outcomes are inferred from candle paths and treat same-bar TP/SL collisions conservatively
  3. screen results are recent-sample diagnostics, not walk-forward validation

## Research and validation summary

This notebook reuses the public REST PMM screener structure for universe discovery and microstructure gating, then adds
MACD-BB signal extraction using the repo's `macd_bb_v1` indicator logic. Strategy quality metrics are based on recent
signal events extracted from public candles and evaluated with an approximate TP / SL / time-limit barrier.

### Market-data gates

- quote-volume, spread, top-of-book, depth, trade recency, and candle-quality checks are inherited from the public PMM screener stack
- strategy metrics are added **after** microstructure enrichment and combined into the final `screen_score`

### Strategy settings used for diagnostics

```yaml
{yaml.safe_dump(asdict(strategy_cfg), sort_keys=False)}
```

### Controller template settings used for exports

```yaml
{yaml.safe_dump(asdict(controller_template_cfg), sort_keys=False)}
```

## Selected pairs

{selected_md}

## Critical issues and live blockers

- Passing the screener means the pair looks usable for **research** and possibly **paper trading**, not live deployment.
- Net-edge figures subtract observed spread only. They do **not** include venue-specific fee schedules unless you add them.
- Spot deployments should assume **long-only** unless you already hold inventory for sell-side management or use a perpetual connector.
- The controller templates are starter YAMLs. Exchange rules in the patch file remain estimates unless you verify them against the live connector.

## Rejection summary

{reject_md}

## Controller dictionary fields used

The controller templates were built from `controller_yml_data_dictionary.md` rows for `macd_bb_v1`:

{dict_md}

## Exact next actions

1. Verify exchange trading rules (`price_tick`, `amount_step`, `min_notional`) against the live connector before paper trading.
2. Increase `CANDLE_LIMIT` and re-run if signal counts are too sparse to trust.
3. Paper trade the exported controllers first and compare realized fills against spread-based net-edge estimates.
4. Do not treat this notebook as a substitute for walk-forward backtesting or exchange-specific execution testing.
"""
    return textwrap.dedent(report).strip() + "\n"


def export_macd_bb_artifacts(
    run: ScreeningRun,
    output_dir: str,
    base_url: str,
    trade_limit: int,
    controller_dict_path: str | Path,
    strategy_cfg: StrategyScreenConfig,
    controller_template_cfg: ControllerTemplateConfig,
) -> MACDBBArtifactPaths:
    base_paths = export_screening_artifacts(
        run,
        output_dir=str(output_dir),
        base_url=base_url,
        trade_limit=int(trade_limit),
    )
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)

    controller_dict_df = read_controller_dictionary_md(controller_dict_path, "macd_bb_v1")
    controller_templates = build_controller_templates(run, controller_template_cfg, controller_dict_df)
    controller_summary = build_controller_summary(run)
    symbol_metadata = build_symbol_metadata(run)
    screening_report = build_screening_report(run, controller_dict_df, strategy_cfg, controller_template_cfg)

    controller_templates_path = root / "controller_templates.yaml"
    controller_summary_path = root / "controller_summary.csv"
    symbol_metadata_path = root / "symbol_metadata.json"
    screening_report_path = root / "screening_report.md"
    run_metadata_path = root / "run_metadata.json"

    with open(controller_templates_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(controller_templates, f, sort_keys=False, allow_unicode=True)
    controller_summary.to_csv(controller_summary_path, index=False)
    with open(symbol_metadata_path, "w", encoding="utf-8") as f:
        json.dump(symbol_metadata, f, indent=2)
    screening_report_path.write_text(screening_report, encoding="utf-8")

    metadata = run.metadata()
    metadata["strategy_screen_config"] = asdict(strategy_cfg)
    metadata["controller_template_config"] = asdict(controller_template_cfg)
    metadata["controller_dictionary_path"] = str(controller_dict_path)
    with open(run_metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return MACDBBArtifactPaths(
        root_dir=str(root),
        universe_csv=base_paths.universe_csv,
        shortlist_csv=base_paths.shortlist_csv,
        final_csv=base_paths.final_csv,
        selected_csv=base_paths.selected_csv,
        selected_pairs_txt=base_paths.selected_pairs_txt,
        selected_pairs_json=base_paths.selected_pairs_json,
        symbol_metadata_json=str(symbol_metadata_path),
        candle_ingestor_manifest_yaml=base_paths.candle_ingestor_manifest_yaml,
        exchange_rules_patch_yaml=base_paths.exchange_rules_patch_yaml,
        controller_templates_yaml=str(controller_templates_path),
        controller_summary_csv=str(controller_summary_path),
        screening_report_md=str(screening_report_path),
        run_metadata_json=str(run_metadata_path),
    )


Repo root: /quants-lab
PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic
Editable install ok: True


## 1. Configuration

Edit the universe, thresholds, and output path here. The controller defaults are loaded from
`controller_yml_data_dictionary.md` for `macd_bb_v1`, while the market-data gates stay close to the PMM Dynamic
public screener defaults for the venue.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
# Keep 5m by default. NonKYC public candles in the repo helper start at 5m.
QUOTE_ASSET = "*"
INTERVAL = "5m"
UNIVERSE_TOP_K = 800
FINAL_TOP_N = 20
CANDLE_LIMIT = 480
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

SIDE_MODE = "long_only"

MIN_SIGNAL_EVENTS = 3
MIN_EVENTS_PER_DAY = 0.0
MIN_HIT_RATE = 0.0
MIN_MEAN_NET_EDGE_BPS = 0.0
MIN_PROFIT_FACTOR_PROXY = 0.95
MAX_AMBIGUOUS_RATE = 0.60
STRATEGY_SCORE_WEIGHT = 0.40

INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

CONTROLLER_TOTAL_AMOUNT_QUOTE = 100.0
CONTROLLER_MAX_EXECUTORS_PER_SIDE = 1
CONTROLLER_LEVERAGE = 1
CONTROLLER_POSITION_MODE = "HEDGE"
CONTROLLER_TAKE_PROFIT_ORDER_TYPE = 2
CONTROLLER_TRAILING_STOP = None

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "macd_bb" / "nonkyc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

controller_dict_df = read_controller_dictionary_md(DICT_PATH, "macd_bb_v1")
controller_defaults = controller_defaults_from_dictionary(controller_dict_df)

cfg = default_nonkyc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative thin-venue defaults.
cfg.min_quote_volume_24h = 50_000.0
cfg.max_spread_bps = 180.0
cfg.min_top_of_book_quote = 5.0
cfg.min_depth_10bps_quote = 0.0
cfg.min_depth_50bps_quote = 0.0
cfg.min_depth_1xspread_quote = 0.0
cfg.max_last_trade_age_sec = 3600.0
cfg.min_recent_trade_count = 30
cfg.min_candle_count = 360
cfg.min_candle_coverage_ratio = 0.92
cfg.max_zero_volume_fraction = 0.35
cfg.min_natr_bps = 15.0
cfg.max_natr_bps = 600.0
cfg.fallback_if_empty = True

strategy_cfg = StrategyScreenConfig(
    interval=INTERVAL,
    interval_seconds=cfg.interval_seconds(),
    bb_length=int(controller_defaults.get("bb_length", 100)),
    bb_std=float(controller_defaults.get("bb_std", 2.0)),
    bb_long_threshold=float(controller_defaults.get("bb_long_threshold", 0.0)),
    bb_short_threshold=float(controller_defaults.get("bb_short_threshold", 1.0)),
    macd_fast=int(controller_defaults.get("macd_fast", 21)),
    macd_slow=int(controller_defaults.get("macd_slow", 42)),
    macd_signal=int(controller_defaults.get("macd_signal", 9)),
    cooldown_time=int(controller_defaults.get("cooldown_time", 300)),
    take_profit=float(controller_defaults.get("take_profit", 0.02)),
    stop_loss=float(controller_defaults.get("stop_loss", 0.03)),
    time_limit_sec=int(controller_defaults.get("time_limit", 2700)),
    side_mode=SIDE_MODE,
    min_signal_events=MIN_SIGNAL_EVENTS,
    min_events_per_day=MIN_EVENTS_PER_DAY,
    min_hit_rate=MIN_HIT_RATE,
    min_mean_net_edge_bps=MIN_MEAN_NET_EDGE_BPS,
    min_profit_factor_proxy=MIN_PROFIT_FACTOR_PROXY,
    max_ambiguous_rate=MAX_AMBIGUOUS_RATE,
    strategy_score_weight=STRATEGY_SCORE_WEIGHT,
)

controller_template_cfg = ControllerTemplateConfig(
    connector_name=cfg.connector,
    candles_connector=cfg.connector,
    interval=INTERVAL,
    total_amount_quote=float(CONTROLLER_TOTAL_AMOUNT_QUOTE),
    max_executors_per_side=int(CONTROLLER_MAX_EXECUTORS_PER_SIDE),
    cooldown_time=int(controller_defaults.get("cooldown_time", 300)),
    leverage=int(CONTROLLER_LEVERAGE),
    position_mode=str(CONTROLLER_POSITION_MODE),
    stop_loss=float(controller_defaults.get("stop_loss", strategy_cfg.stop_loss)),
    take_profit=float(controller_defaults.get("take_profit", strategy_cfg.take_profit)),
    time_limit=int(controller_defaults.get("time_limit", strategy_cfg.time_limit_sec)),
    take_profit_order_type=int(CONTROLLER_TAKE_PROFIT_ORDER_TYPE),
    trailing_stop=CONTROLLER_TRAILING_STOP,
    bb_length=int(controller_defaults.get("bb_length", 100)),
    bb_std=float(controller_defaults.get("bb_std", 2.0)),
    bb_long_threshold=float(controller_defaults.get("bb_long_threshold", 0.0)),
    bb_short_threshold=float(controller_defaults.get("bb_short_threshold", 1.0)),
    macd_fast=int(controller_defaults.get("macd_fast", 21)),
    macd_slow=int(controller_defaults.get("macd_slow", 42)),
    macd_signal=int(controller_defaults.get("macd_signal", 9)),
)

dict_preview = controller_dict_df.loc[
    controller_dict_df["field_path"].isin(
        [
            "controller_name",
            "controller_type",
            "total_amount_quote",
            "connector_name",
            "trading_pair",
            "max_executors_per_side",
            "cooldown_time",
            "leverage",
            "position_mode",
            "stop_loss",
            "take_profit",
            "time_limit",
            "take_profit_order_type",
            "candles_connector",
            "candles_trading_pair",
            "interval",
            "bb_length",
            "bb_std",
            "bb_long_threshold",
            "bb_short_threshold",
            "macd_fast",
            "macd_slow",
            "macd_signal",
        ]
    ),
    ["field_path", "default", "what_it_does"],
].reset_index(drop=True)

print("Market screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))

print("MACD-BB dictionary defaults used")
display(dict_preview)

print("Strategy screen config")
display(pd.Series(asdict(strategy_cfg)).to_frame("value"))

print("Controller template config")
display(pd.Series(asdict(controller_template_cfg)).to_frame("value"))

print(f"Output root: {OUTPUT_ROOT}")


Market screener config


,value
connector,nonkyc
quote_asset,*
interval,5m
universe_top_k,800
final_top_n,20
candle_limit,480
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.2
timeout_seconds,30.0


MACD-BB dictionary defaults used


,field_path,default,what_it_does
0,controller_name,macd_bb_v1,Controller module name. This must match the controller implementation and is used when loading the controller from the YAML file.
1,controller_type,directional_trading,"Controller family used for grouping/loading the controller. In this repo the main families are generic, directional_trading, and market_making. Allowed valu..."
2,total_amount_quote,100,"Total budget in quote currency that the controller uses when sizing positions, grids, or quote-side order allocations."
3,connector_name,binance_perpetual,Exchange/connector name that the controller uses for trading or market data.
4,trading_pair,WLD-USDT,"Market symbol the controller trades or monitors, in `BASE-QUOTE` form."
5,max_executors_per_side,2,Maximum number of concurrent executors the directional controller may keep open on each side (long/buy or short/sell).
6,cooldown_time,60 * 5,"Minimum wait time between signal/executor creations or rebalance actions, depending on the controller."
7,leverage,1,Leverage applied when the connector supports perpetual or margin trading.
8,position_mode,HEDGE,"Perpetual position mode. Typical values are `HEDGE` and `ONEWAY`. Allowed values / format: HEDGE, ONEWAY."
9,stop_loss,0.03,Relative loss threshold that closes an executor/position when exceeded.


Strategy screen config


,value
interval,5m
interval_seconds,300
bb_length,100
bb_std,2.0
bb_long_threshold,0.0
bb_short_threshold,1.0
macd_fast,21
macd_slow,42
macd_signal,9
cooldown_time,300


Controller template config


,value
connector_name,nonkyc
candles_connector,nonkyc
interval,5m
total_amount_quote,100.0
max_executors_per_side,1
cooldown_time,300
leverage,1
position_mode,HEDGE
stop_loss,0.03
take_profit,0.02


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any
per-pair enrichment calls are made.


In [3]:
screener = NonKYCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,nonkyc,*,5m,345,345


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,XLM-USDT,XLM/USDT,1.211292e+05,6.036825,76.519197,1.000000e-04,0.010000,active
1,ETH-USDT,ETH/USDT,8.828831e+06,38.885158,76.478007,1.000000e-02,0.000010,active
2,FF-USDT,FF/USDT,9.916696e+04,9.883516,76.054157,1.000000e-05,0.010000,active
3,NKYC-USDT,NKYC/USDT,1.539981e+05,17.619170,75.874523,1.000000e-06,0.000100,active
4,USDC-USDT,USDC/USDT,2.330981e+06,48.987753,75.660692,1.000000e-04,0.010000,active
5,SUN-USDT,SUN/USDT,8.274200e+04,2.927486,75.476354,1.000000e-06,0.010000,active
6,TRX-USDT,TRX/USDT,4.728368e+05,38.634900,74.797483,1.000000e-04,0.010000,active
7,ONDO-USDT,ONDO/USDT,4.661800e+05,45.293274,74.053267,1.000000e-05,0.010000,active
8,SHIB-USDT,SHIB/USDT,2.036378e+05,41.797283,73.867404,1.000000e-09,1.000000,active
9,XMR-USDT,XMR/USDT,1.899265e+06,61.272633,73.806763,1.000000e-02,0.001000,active


Shortlist for detailed enrichment: 345


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,XLM-USDT,XLM/USDT,1.211292e+05,6.036825,76.519197,1.000000e-04,0.010000,active
1,ETH-USDT,ETH/USDT,8.828831e+06,38.885158,76.478007,1.000000e-02,0.000010,active
2,FF-USDT,FF/USDT,9.916696e+04,9.883516,76.054157,1.000000e-05,0.010000,active
3,NKYC-USDT,NKYC/USDT,1.539981e+05,17.619170,75.874523,1.000000e-06,0.000100,active
4,USDC-USDT,USDC/USDT,2.330981e+06,48.987753,75.660692,1.000000e-04,0.010000,active
5,SUN-USDT,SUN/USDT,8.274200e+04,2.927486,75.476354,1.000000e-06,0.010000,active
6,TRX-USDT,TRX/USDT,4.728368e+05,38.634900,74.797483,1.000000e-04,0.010000,active
7,ONDO-USDT,ONDO/USDT,4.661800e+05,45.293274,74.053267,1.000000e-05,0.010000,active
8,SHIB-USDT,SHIB/USDT,2.036378e+05,41.797283,73.867404,1.000000e-09,1.000000,active
9,XMR-USDT,XMR/USDT,1.899265e+06,61.272633,73.806763,1.000000e-02,0.001000,active


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then computes MACD-BB
signal diagnostics using the repo controller logic: Bollinger `%B`, MACD line, and MACD histogram. The final
`screen_score` blends market microstructure quality with recent strategy evidence.


In [4]:
run = screen_macd_bb_from_universe(
    screener=screener,
    universe=universe,
    market_cfg=cfg,
    strategy_cfg=strategy_cfg,
)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float((final_df["passed_filters"].mean()) if len(final_df) else 0.0),
        "selection_mode": run.selection_mode,
    }
])
display(status_counts)
display(pd.Series(run.notes, name="note").to_frame())

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "microstructure_screen_score",
        "strategy_score",
        "passed_filters",
        "strategy_signal_count",
        "strategy_hit_rate",
        "strategy_mean_barrier_return_net_bps",
        "strategy_profit_factor_proxy",
        "strategy_current_signal",
        "strategy_recommended_side",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


GET https://api.nonkyc.io/api/v2/market/trades?symbol=SATOX%2FUSDT&limit=500 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=PDOGE%2FBNB&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=PDOGE%2FBNB&limit=200 failed on attempt 2/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=TRX%2FBTC&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
/opt/conda/envs/quants-lab/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/conda/envs/quants-lab/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


,enriched_rows,selected_rows,pass_rate,selection_mode
0,345,20,0.0,fallback_ranked


,note
0,Research / paper-trade triage only. Passing the screener does not imply live readiness.
1,"Strategy metrics are approximate candle-path diagnostics, not full execution-aware backtests."
2,Round-trip fees are not modeled; net-edge figures subtract observed spread only.
3,Default side_mode is long_only because these public screeners use spot market data.
4,WARNING: No pairs passed strict hard gates. Selected set is fallback_ranked. Review rejection_reason carefully.


,trading_pair,screen_score,microstructure_screen_score,strategy_score,passed_filters,strategy_signal_count,strategy_hit_rate,strategy_mean_barrier_return_net_bps,strategy_profit_factor_proxy,strategy_current_signal,strategy_recommended_side,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,ALGO-USDC,77.764718,86.186670,65.131790,False,1,1.000000,-25.715440,inf,0,long,48741.8173,73.619632,1552.342624,0.000000,200,8.891287,480,0.995851,0.000000,19.785198,0.099678,quote_volume_24h<50000; strategy_signal_count<3; strategy_mean_net_edge_bps<0
1,ARB-USDT,77.704547,85.750863,65.635075,False,2,1.000000,-45.213857,inf,0,long,52796.3534,67.264574,169.259377,0.000000,200,3.537505,480,0.993789,0.000000,22.908250,0.104925,strategy_signal_count<3; strategy_mean_net_edge_bps<0
2,AAVE-USDT,77.579691,86.713101,63.879575,False,4,0.500000,-75.865937,1.223985,0,long,80001.2507,78.399010,385.467520,0.000000,200,18.886153,480,0.997921,0.000000,22.968909,0.152327,strategy_mean_net_edge_bps<0
3,ETC-USDT,75.850468,82.553894,65.795329,False,1,1.000000,-24.436435,inf,0,long,45108.6453,61.842919,0.185380,0.000000,200,15.954955,480,1.000000,0.000000,25.787292,0.048951,quote_volume_24h<50000; top_of_book_quote<5; strategy_signal_count<3; strategy_mean_net_edge_bps<0
4,LTC-USDT,75.250148,81.363808,66.079658,False,2,1.000000,-43.039159,inf,0,long,617564.1784,59.779563,0.032022,0.000000,200,11.036388,480,0.965795,0.012500,22.378710,0.064621,top_of_book_quote<5; strategy_signal_count<3; strategy_mean_net_edge_bps<0
5,SAL-USDT,74.908251,78.099300,70.121677,False,1,1.000000,22.201653,inf,0,long,40521.7725,0.585840,0.639219,0.639219,200,117.468539,480,0.971660,0.000000,68.952616,0.037529,quote_volume_24h<50000; top_of_book_quote<5; strategy_signal_count<3
6,SCASH-USDT,74.048514,76.892103,69.783130,False,7,1.000000,-46.160373,inf,0,long,10587.6557,84.737957,5.277956,0.000000,200,10.307478,480,0.993789,0.000000,52.331170,0.054574,quote_volume_24h<50000; strategy_mean_net_edge_bps<0
7,ZEC-USDT,73.711006,76.410500,69.661766,False,2,1.000000,-22.856242,inf,0,short,40142.8417,84.677988,0.745010,0.000000,200,18.026071,480,0.941176,0.025000,35.966033,0.067920,quote_volume_24h<50000; top_of_book_quote<5; strategy_signal_count<3; strategy_mean_net_edge_bps<0
8,ARB-USDC,73.063490,77.163623,66.913291,False,3,0.666667,-71.165063,3.003308,0,long,14883.9240,78.519349,0.120825,0.000000,200,8.369746,480,0.997921,0.000000,22.439909,0.102128,quote_volume_24h<50000; top_of_book_quote<5; strategy_mean_net_edge_bps<0
9,ADA-USDT,73.030611,78.681566,64.554178,False,3,0.333333,-58.403872,1.500761,0,long,8187.7693,61.112243,0.176184,0.000000,200,13.508671,480,1.000000,0.000000,29.372778,0.106812,quote_volume_24h<50000; top_of_book_quote<5; strategy_mean_net_edge_bps<0


## 4. Diagnostics

Review the selected pairs, rejection reasons, and whether the recent signal sample is strong enough to justify
moving the pair forward into deeper backtesting or paper trading.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {len(passed)} | Rejected: {len(rejected)} | Selection mode: {run.selection_mode}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "strategy_score",
                    "strategy_signal_count",
                    "strategy_hit_rate",
                    "strategy_mean_barrier_return_net_bps",
                    "strategy_profit_factor_proxy",
                    "strategy_current_signal",
                    "strategy_recommended_side",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if "strategy_current_signal" in final_df.columns:
        signal_state = (
            final_df["strategy_current_signal"]
            .map({1: "long_now", -1: "short_now", 0: "flat"})
            .fillna("unknown")
            .value_counts()
            .rename_axis("current_signal_state")
            .to_frame("count")
        )
        display(signal_state)

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: 0 | Rejected: 345 | Selection mode: fallback_ranked


,count
current_signal_state,
flat,343
long_now,2


,count
rejection_reason,
top_of_book_quote<5,306
quote_volume_24h<50000,303
strategy_signal_count<3,302
missing_strategy_profit_factor_proxy,226
missing_strategy_mean_net_edge_bps,206
coverage_ratio<0.92,201
strategy_mean_net_edge_bps<0,126
natr_bps_mean<15,71
strategy_profit_factor_proxy<0.95,39


## 5. Export artifacts

This writes CSV, JSON, YAML, and Markdown artifacts under the chosen output root. In addition to the usual
selected-pairs / manifest / rules-patch bundle, it exports `controller_templates.yaml` and `controller_summary.csv`
for `macd_bb_v1`.


In [6]:
artifact_paths = export_macd_bb_artifacts(
    run=run,
    output_dir=str(OUTPUT_ROOT),
    base_url=NONKYC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
    controller_dict_path=DICT_PATH,
    strategy_cfg=strategy_cfg,
    controller_template_cfg=controller_template_cfg,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/candle_ingestor_manifest.yaml
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/macd_bb/nonkyc/20260328_044628/exchange_rules_patch.yaml


## 6. Merge into candle ingestion

The artifacts below follow the same selected-pairs / manifest / rules-patch pattern used by the PMM Dynamic
public screener, with an additional `controller_templates.yaml` bundle for `directional_trading.macd_bb_v1`.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())

print("\nController templates (starter YAML)")
print("-" * 80)
print(Path(artifact_paths.controller_templates_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
ALGO-USDC
ARB-USDT
AAVE-USDT
ETC-USDT
LTC-USDT
SAL-USDT
SCASH-USDT
ZEC-USDT
ARB-USDC
ADA-USDT
EQPAY-USDT
SHIB-USDT
UNI-USDT
BTC-USDC
CAKE-USDC
PEP-DOGE
XLM-USDT
PEP-USDC
ARRR-USDT
PEP-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  nonkyc:
    enabled: true
    base_url: https://api.nonkyc.io/api/v2
    pairs:
    - ALGO/USDC
    - ARB/USDT
    - AAVE/USDT
    - ETC/USDT
    - LTC/USDT
    - SAL/USDT
    - SCASH/USDT
    - ZEC/USDT
    - ARB/USDC
    - ADA/USDT
    - EQPAY/USDT
    - SHIB/USDT
    - UNI/USDT
    - BTC/USDC
    - CAKE/USDC
    - PEP/DOGE
    - XLM/USDT
    - PEP/USDC
    - ARRR/USDT
    - PEP/USDT
    int